In [3]:
import duckdb
from pathlib import Path

def logical_compare_parquet(dir1, dir2):
    dir1 = str(Path(dir1))
    dir2 = str(Path(dir2))
    con = duckdb.connect()

    # List .parquet files in both dirs
    names1 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir1}/*.parquet')").fetchall()}
    names2 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir2}/*.parquet')").fetchall()}
    common = sorted(names1 & names2)

    only1 = sorted(names1 - names2)
    only2 = sorted(names2 - names1)
    print(f"Only in {dir1}: {only1}")
    print(f"Only in {dir2}: {only2}")

    all_ok = True
    for name in common:
        path1 = f"{dir1}/{name}"
        path2 = f"{dir2}/{name}"

        # Read schemas
        s1 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path1}')").fetchdf()
        s2 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path2}')").fetchdf()
        schema_equal = s1[['column_name','column_type']].equals(s2[['column_name','column_type']])

        # Row counts
        n1 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path1}')").fetchone()[0]
        n2 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path2}')").fetchone()[0]

        # Set difference both ways (assumes rows are comparable; if there’s no stable key,
        # this treats the entire row as a tuple)
        diff12 = con.execute(
            f"SELECT * FROM parquet_scan('{path1}') EXCEPT ALL SELECT * FROM parquet_scan('{path2}')"
        ).fetchone()
        diff21 = con.execute(
            f"SELECT * FROM parquet_scan('{path2}') EXCEPT ALL SELECT * FROM parquet_scan('{path1}')"
        ).fetchone()

        # If EXCEPT returns at least one row, DuckDB returns a row; we only need to know emptiness.
        has_diff12 = diff12 is not None
        has_diff21 = diff21 is not None

        if schema_equal and n1 == n2 and not has_diff12 and not has_diff21:
            print(f"[OK]   {name}: same schema, same rows ({n1} rows).")
        else:
            all_ok = False
            print(f"[DIFF] {name}: schema_equal={schema_equal}, rowsA={n1}, rowsB={n2}, "
                  f"A\\B_diff={has_diff12}, B\\A_diff={has_diff21}")

    if all_ok and not only1 and not only2:
        print("✅ All matching files have identical table contents.")
    else:
        print("❗Content differences found (see above).")

# Example:
path_a = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/Ischemic_Stroke"
path_b = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte_first/Ischemic_Stroke"

logical_compare_parquet(path_a,path_b)


Only in /data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/Ischemic_Stroke: []
Only in /data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte_first/Ischemic_Stroke: []
[OK]   held_out.parquet: same schema, same rows (49438 rows).
[OK]   train.parquet: same schema, same rows (78888 rows).
[OK]   tuning.parquet: same schema, same rows (19809 rows).
✅ All matching files have identical table contents.


In [8]:
import pandas as pd
from pathlib import Path
base_dir = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_0.1")
# train_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform/data/train")
# for i in range(0,10):
#     train_cohort = pd.read_parquet(train_path/f"{i}.parquet")
#     print(train_cohort["subject_id"])
# meta_path = base_dir/"metadata"/"subject_splits.parquet"
# meta_path = base_dir/"metadata"/"vocabulary_cache"/"mapping"/"data.parquet"
id_list = []
data_dir = base_dir/"data"
for subdir in data_dir.iterdir():
    if not subdir.is_dir():
        continue
    for f in subdir.glob("*.parquet"):
        raw_file = pd.read_parquet(f)
        id_list.extend(raw_file["subject_id"].unique())
        # break
        # id_list.extend(raw_file)
    # break
print(len(id_list))
# meta_data = pd.read_parquet(meta_path)
# print(meta_data)

690298


In [ ]:
result_path = "/data/results/zj2398/fomoh/mimic/results"

import os
import json
import re
import pandas as pd
from pathlib import Path

def rename_files_and_dirs(base_dir):
    """Remove '_linear_probing_' from all file and directory names."""
    base_path = Path(base_dir)
    
    for disease_dir in base_path.iterdir():
        if not disease_dir.is_dir():
            continue
        
        # Collect all items to rename (files and dirs)
        items_to_rename = []
        for item in disease_dir.iterdir():
            if "_linear_probing_" in item.name:
                new_name = item.name.replace("_linear_probing_", "_")
                items_to_rename.append((item, disease_dir / new_name))
        
        # Rename (do directories last to avoid path issues)
        # Sort: files first (has extension), then directories
        items_to_rename.sort(key=lambda x: x[0].is_dir())
        
        for old_path, new_path in items_to_rename:
            print(f"Renaming: {old_path.name} -> {new_path.name}")
            old_path.rename(new_path)

def extract_results(base_dir):
    """Extract ROC-AUC scores from the largest-numbered subdirectories."""
    base_path = Path(base_dir)
    results = {}  # {disease: {model: (mean, std)}}
    
    for disease_dir in base_path.iterdir():
        if not disease_dir.is_dir():
            continue
        
        disease_name = disease_dir.name
        results[disease_name] = {}
        
        # Find all model subdirectories (after renaming, pattern is model_*)
        model_dirs = {}  # {model_name: {num: path}}
        
        for item in disease_dir.iterdir():
            if item.is_dir():
                # Match pattern: llama_* or mamba_* (after renaming)
                # Or original: llama_linear_probing_* or mamba_linear_probing_*
                match = re.match(r'^(llama|mamba)(?:_linear_probing)?_(\d+)$', item.name)
                if match:
                    model_name = match.group(1)
                    num = int(match.group(2))
                    if model_name not in model_dirs:
                        model_dirs[model_name] = {}
                    model_dirs[model_name][num] = item
        
        # For each model, find the largest numbered directory
        for model_name, num_paths in model_dirs.items():
            if not num_paths:
                continue
            
            max_num = max(num_paths.keys())
            target_dir = num_paths[max_num]
            
            # Read results_boot.json
            json_path = target_dir / "results_boot.json"
            if json_path.exists():
                with open(json_path, 'r') as f:
                    data = json.load(f)
                
                mean_auc = data.get("mean_roc_auc_score", None)
                std_auc = data.get("std_roc_auc_score", None)
                results[disease_name][model_name] = (mean_auc, std_auc)
                print(f"{disease_name}/{model_name} (n={max_num}): {mean_auc:.4f} ± {std_auc:.4f}")
            else:
                print(f"Warning: {json_path} not found")
    
    return results

def create_csv(results, output_path):
    """Create CSV with rows=models, columns=diseases, values=mean(std)."""
    # Get all diseases and models
    diseases = sorted(results.keys())
    models = set()
    for disease_data in results.values():
        models.update(disease_data.keys())
    models = sorted(models)
    
    # Build dataframe
    data = {}
    for disease in diseases:
        col = []
        for model in models:
            if model in results[disease]:
                mean, std = results[disease][model]
                col.append(f"{mean:.4f}({std:.4f})")
            else:
                col.append("")
        data[disease] = col
    
    df = pd.DataFrame(data, index=models)
    df.index.name = "model"
    df.to_csv(output_path)
    print(f"\nSaved to {output_path}")
    return df

def main(base_dir, output_csv="roc_auc_results.csv", do_rename=True):
    # Step 1: Rename files/directories
    if do_rename:
        print("=== Renaming files and directories ===")
        rename_files_and_dirs(base_dir)
        print()
    
    # Step 2: Extract results
    print("=== Extracting results ===")
    results = extract_results(base_dir)
    
    # Step 3: Create CSV
    print("\n=== Creating CSV ===")
    df = create_csv(results, output_csv)
    print(df)
    
    return df

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument("base_dir", help="Base directory containing disease subdirectories")
    parser.add_argument("-o", "--output", default="roc_auc_results.csv", help="Output CSV path")
    parser.add_argument("--no-rename", action="store_true", help="Skip renaming step")
    args = parser.parse_args()
    
    main(args.base_dir, args.output, do_rename=not args.no_rename)

SyntaxError: invalid syntax (2392180866.py, line 1)

In [1]:
from pathlib import Path
import pandas as pd
def rename_files_and_dirs(base_dir):
    """Remove '_730_' from all file and directory names."""
    base_path = Path(base_dir)
    
    for disease_dir in base_path.iterdir():
        if not disease_dir.is_dir():
            continue
        
        # Collect all items to rename (files and dirs)
        items_to_rename = []
        for item in disease_dir.iterdir():
            if "motor_730" in item.name:
                new_name = item.name.replace("motor_730", "motor")
                items_to_rename.append((item, disease_dir / new_name))
        
        # Rename (do directories last to avoid path issues)
        # Sort: files first (has extension), then directories
        items_to_rename.sort(key=lambda x: x[0].is_dir())
        
        for old_path, new_path in items_to_rename:
            print(f"Renaming: {old_path.name} -> {new_path.name}")
            old_path.rename(new_path)

result_path = "/shared/share_mala/zj2398/mimic/task/phenotype_task/few_shot_results"
rename_files_and_dirs(result_path)

Renaming: motor_730_100.parquet -> motor_100.parquet
Renaming: motor_730_1000.parquet -> motor_1000.parquet
Renaming: motor_730_10000.parquet -> motor_10000.parquet
Renaming: motor_730_100000.parquet -> motor_100000.parquet
Renaming: motor_730_1000 -> motor_1000
Renaming: motor_730_100000 -> motor_100000
Renaming: motor_730_10000 -> motor_10000
Renaming: motor_730_100 -> motor_100
Renaming: motor_730_100.parquet -> motor_100.parquet
Renaming: motor_730_1000.parquet -> motor_1000.parquet
Renaming: motor_730_10000.parquet -> motor_10000.parquet
Renaming: motor_730_100000.parquet -> motor_100000.parquet
Renaming: motor_730_1000 -> motor_1000
Renaming: motor_730_100000 -> motor_100000
Renaming: motor_730_10000 -> motor_10000
Renaming: motor_730_100 -> motor_100
Renaming: motor_730_100.parquet -> motor_100.parquet
Renaming: motor_730_1000.parquet -> motor_1000.parquet
Renaming: motor_730_10000.parquet -> motor_10000.parquet
Renaming: motor_730_100000.parquet -> motor_100000.parquet
Renaming

In [ ]:
import pandas as pd
from pathlib import Path
cumc_split = pd.read_csv("/data2/processed_datasets/zj2398/femr/cumc/main_split.csv")
train_split = cumc_split[cumc_split["split_name"]=="train"]["subject_id"].to_list()
test_split = cumc_split[cumc_split["split_name"]=="test"]["subject_id"].to_list()
# outcome_cohort_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/patient_outcomes_clean")
outcome_cohort_path =  Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/binary")

def check_list(list1,list2):
    set1 = set(list1)
    set2 = set(list2)
    unique1 =  set1 - set2
    unique2 = set2 - set1
    print(f"split has {len(unique1)} elements, whole split has {len(unique2)} elements")
    
# for task in ["death","long_los","readmission"]:
# for task in ["AMI","Celiac","Ischemic_Stroke", "MASLD"]:
for task in ["stroke"]:
    task_train = pd.read_parquet(outcome_cohort_path/task/"train.parquet")["subject_id"].to_list()
    task_tuning = pd.read_parquet(outcome_cohort_path/task/"tuning.parquet")["subject_id"].to_list()
    task_test = pd.read_parquet(outcome_cohort_path/task/"held_out.parquet")["subject_id"].to_list()
    # print(task_train)
    check_list(task_train,train_split)
    check_list(task_tuning,train_split)
    check_list(task_test,test_split)


split has 0 elements, whole split has 4683213 elements
split has 0 elements, whole split has 4698113 elements
split has 0 elements, whole split has 2005042 elements
split has 0 elements, whole split has 4679763 elements
split has 0 elements, whole split has 4697332 elements
split has 0 elements, whole split has 2002863 elements
split has 0 elements, whole split has 4688003 elements
split has 0 elements, whole split has 4699190 elements
split has 0 elements, whole split has 2007769 elements
split has 0 elements, whole split has 4683745 elements
split has 0 elements, whole split has 4698295 elements
split has 0 elements, whole split has 2005327 elements


In [2]:
from pathlib import Path
import pandas as pd
import meds_reader
# mimic_split_path = Path("/data/raw_data/mimic/files/mimiciv/meds_v0.6/3.1/MEDS_cohort/metadata")
mimic_split_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform/metadata")
meds_reader_path = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_meds_reader"
mimic_split = pd.read_parquet(mimic_split_path/"subject_splits.parquet")
# for split_name in ['train','tuning','held_out']:
#     print(split_name,len(mimic_split[mimic_split["split"]==split_name]))

split_events = {"train":0,"tuning":0,"held_out":0}
train_list = list(mimic_split[mimic_split["split"]=="train"]["subject_id"])
tuning_list = list(mimic_split[mimic_split["split"]=="tuning"]["subject_id"])
held_out_list = list(mimic_split[mimic_split["split"]=="held_out"]["subject_id"])

meds_database = meds_reader.SubjectDatabase(meds_reader_path)
for subject_id in meds_database:
    if subject_id in train_list:
        split_events["train"] += len(meds_database[subject_id].events)
    elif  subject_id in tuning_list:
        split_events["tuning"] += len(meds_database[subject_id].events)
    elif  subject_id in held_out_list:
        split_events["held_out"] += len(meds_database[subject_id].events)
    else:
        raise Exception(f"{subject_id} not found")
for key,item in split_events:
    print(key,item)

KeyboardInterrupt: 

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.